# MOUSE · Multimodální embeddingy v1.2
## Jeden notebook, tři karty, bezpečné pokračování po přerušení

Zachovává původní experiment Qwen3-VL-Embedding-2B:
- Myš, Kocour, Ryba
- samostatný český text
- samostatná ilustrace
- text a ilustrace společně
- křížová podobnost text × obraz, pořadí párování, tři matice podobností, srovnání společné reprezentace, PCA, délky textů, metadata a ZIP.

**Co je nově:** vstupy, cache modelu a každý jednotlivý embedding se ukládají na Google Drive. Každý embedding má kontrolní otisk modelu a vstupu. Po přerušení se hotové a odpovídající vektory načtou z Drive, místo aby se znovu počítaly. Pokud se změní model nebo vstupní text/obrázek, starý vektor se nepoužije.

Po odpojení se musí znovu připojit běhové prostředí, připojit Drive a načíst model do RAM. To je nutné, ale samotné embeddingy se znovu počítat nemají.

Notebook je jeden dokument. Buňky spouštěj postupně. GPU se použije automaticky, pokud je dostupná; jinak poběží CPU.

## ETAPA 1 · Instalace knihoven

In [ ]:
# 1. Instalace pouze potřebných knihoven
import importlib.metadata as metadata
import subprocess, sys

required = {
    "sentence-transformers": ("sentence-transformers", (5, 4, 0)),
    "transformers": ("transformers", (4, 57, 1)),
}
to_install = []
for package, (distribution, minimum) in required.items():
    try:
        installed = metadata.version(distribution)
        parts = tuple(int(x) for x in installed.split(".")[:3] if x.isdigit())
        if parts < minimum:
            to_install.append(f"{package}>={'.'.join(map(str, minimum))}")
            print(f"{package}: {installed}, bude potřeba aktualizovat")
        else:
            print(f"{package}: {installed}, vyhovuje")
    except metadata.PackageNotFoundError:
        to_install.append(f"{package}>={'.'.join(map(str, minimum))}")

if to_install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *to_install])
else:
    print("Potřebné knihovny už jsou nainstalované.")

## ETAPA 2 · Připojení Google Drive a nastavení složek

Tuto buňku spusť při každém novém připojení Colabu. Drive uchovává vstupní soubory, cache Qwenu, checkpointy i výsledky.

In [ ]:
# 2. Připojení Drive a trvalé složky
from google.colab import drive
drive.mount("/content/drive")

import os, io, re, json, time, zipfile, requests, warnings, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
from IPython.display import display, HTML
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

BASE = Path("/content/drive/MyDrive/MOUSE_Multimodal_Embeddings_v1_2")
CACHE_DIR = BASE / "model_cache"
OUTPUT_DIR = BASE / "results"
INPUT_DIR = OUTPUT_DIR / "inputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
for folder in [CACHE_DIR, OUTPUT_DIR, INPUT_DIR, CHECKPOINT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Nastavit cache ještě před importem transformers / sentence-transformers.
os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(CACHE_DIR / "hub")
os.environ["HUGGINGFACE_HUB_CACHE"] = str(CACHE_DIR / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(CACHE_DIR / "transformers")

REPO = "https://raw.githubusercontent.com/artmisanovak-prog/generative_gen/main"
TEXT_DIR = "doma/primarni_data/kriva_pohadka"
IMAGE_DIR = "doma/primarni_data/ilustrace"
MODEL_ID = "Qwen/Qwen3-VL-Embedding-2B"

CARD_FILES = [
    ("mys", "mys", "O MYŠI"),
    ("kocour", "kocour", "O KOCOUROVI"),
    ("ryba", "ryba", "O RYBĚ"),
]

print("Model:", MODEL_ID)
print("Trvalé úložiště:", BASE)
print("Cache modelu:", CACHE_DIR)
print("Checkpointy:", CHECKPOINT_DIR)

## ETAPA 3 · Načtení tří karet

Text se bere od vlastního nadpisu pohádky. Obrázky se opraví podle EXIF orientace. Při dalším spuštění se už stažené vstupy načtou z Drive.

In [ ]:
# 3. Načtení textů a ilustrací
def download_bytes(url):
    response = requests.get(url, timeout=90)
    response.raise_for_status()
    return response.content

def extract_story_from_title(full_text, title_marker):
    # Skutečné znaky BOM a konce řádků, nikoli jejich doslovný zápis.
    lines = full_text.replace("\ufeff", "").splitlines()
    marker = title_marker.casefold()
    for i, line in enumerate(lines):
        if line.strip().casefold().startswith(marker):
            story = "\n".join(lines[i:]).strip()
            if story:
                return story
    raise ValueError(f"Nenalezen nadpis pohádky: {title_marker}")

records = []
errors = []

for image_stem, text_stem, title_marker in CARD_FILES:
    txt_path = INPUT_DIR / f"{image_stem}.txt"
    img_path = INPUT_DIR / f"{image_stem}.jpg"
    text_url = f"{REPO}/{TEXT_DIR}/{text_stem}.txt"
    image_url = f"{REPO}/{IMAGE_DIR}/{image_stem}.jpg"

    try:
        if txt_path.exists() and img_path.exists():
            story_text = txt_path.read_text(encoding="utf-8")
            image_bytes = img_path.read_bytes()
            print(f"NAČTENO Z DRIVE: {image_stem}")
        else:
            raw_text = download_bytes(text_url).decode("utf-8-sig")
            story_text = extract_story_from_title(raw_text, title_marker)
            original_image_bytes = download_bytes(image_url)
            opened = Image.open(io.BytesIO(original_image_bytes))
            image = ImageOps.exif_transpose(opened).convert("RGB")
            buffer = io.BytesIO()
            image.save(buffer, format="JPEG", quality=95)
            image_bytes = buffer.getvalue()
            txt_path.write_text(story_text, encoding="utf-8")
            img_path.write_bytes(image_bytes)
            print(f"STAŽENO A ULOŽENO NA DRIVE: {image_stem}")

        image = ImageOps.exif_transpose(Image.open(io.BytesIO(image_bytes))).convert("RGB")
        records.append({
            "id": image_stem,
            "title": title_marker,
            "text_file": f"{image_stem}.txt",
            "image_file": f"{image_stem}.jpg",
            "text": story_text,
            "image": image,
            "image_bytes": image_bytes,
            "text_url": text_url,
            "image_url": image_url,
        })
        print(f"✓ {image_stem}: {len(story_text)} znaků, obrázek {image.size}")
    except Exception as e:
        errors.append({"id": image_stem, "error": str(e)})
        print(f"CHYBA {image_stem}: {e}")

print("\nNačteno dvojic:", len(records))
if errors:
    display(pd.DataFrame(errors))
if len(records) != len(CARD_FILES):
    raise RuntimeError("Některá z požadovaných dvojic se nenačetla. Nejdřív oprav chyby.")

## Kontrola vstupů

In [ ]:
# 4. Náhled tří načtených karet
for r in records:
    print(f"\n{r['id']} · {r['text_file']} · {r['image_file']}")
    display(r["image"].resize((260, 260)))
    preview = r["text"][:650].replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML(f"<div style='max-width:850px;white-space:pre-wrap'>{preview}{' …' if len(r['text'])>650 else ''}</div>"))

## ETAPA 4 · Načtení Qwen modelu

Při prvním spuštění se model stáhne do cache na Google Drive. Při dalším připojení se z ní znovu načte. Samotné načtení modelu do paměti RAM/VRAM se po odpojení opakuje, protože Colab zruší původní běhové prostředí.

In [ ]:
# 5. Načtení Qwen modelu
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Zařízení:", device)
if device == "cpu":
    print("GPU není přidělené. Výpočet poběží na CPU, pravděpodobně pomaleji.")

model = SentenceTransformer(
    MODEL_ID,
    device=device,
    trust_remote_code=True,
    cache_folder=str(CACHE_DIR)
)
print("Model načten.")

## ETAPA 5 · Obnovitelný výpočet embeddingů

Každá karta a každý režim se ukládají zvlášť. Je jich celkem devět. Po dokončení každého vektoru se uloží:
- samotný vektor,
- kontrolní metadata s modelem a otiskem vstupu.

Při pokračování se vektor použije jen tehdy, pokud souhlasí model i obsah vstupu. Nedokončený nebo neodpovídající checkpoint se přepočítá. Ukládání probíhá atomicky, aby přerušení během zápisu nezanechalo soubor vydávaný za hotový.

In [ ]:
# 6. Checkpointy po jednotlivých kartách a modalitách
def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def item_fingerprint(modality, card):
    h = hashlib.sha256()
    h.update(MODEL_ID.encode("utf-8"))
    h.update(modality.encode("utf-8"))
    if modality in ("text", "joint"):
        h.update(card["text"].encode("utf-8"))
    if modality in ("image", "joint"):
        h.update(sha256_bytes(card["image_bytes"]).encode("ascii"))
    return h.hexdigest()

def checkpoint_paths(modality, card_id):
    stem = f"{modality}_{card_id}"
    return CHECKPOINT_DIR / f"{stem}.npy", CHECKPOINT_DIR / f"{stem}.json"

def save_checkpoint(vector, vector_path, meta_path, metadata):
    tmp_vector = vector_path.with_suffix(".npy.tmp")
    with open(tmp_vector, "wb") as f:
        np.save(f, vector)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp_vector, vector_path)

    tmp_meta = meta_path.with_suffix(".json.tmp")
    tmp_meta.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    os.replace(tmp_meta, meta_path)

def encode_one(modality, card, item):
    vector_path, meta_path = checkpoint_paths(modality, card["id"])
    fingerprint = item_fingerprint(modality, card)

    if vector_path.exists() and meta_path.exists():
        try:
            saved_meta = json.loads(meta_path.read_text(encoding="utf-8"))
            saved_vector = np.load(vector_path)
            valid = (
                saved_meta.get("model") == MODEL_ID
                and saved_meta.get("fingerprint") == fingerprint
                and saved_meta.get("modality") == modality
                and saved_meta.get("card_id") == card["id"]
                and saved_vector.ndim == 1
                and np.isfinite(saved_vector).all()
            )
            if valid:
                print(f"PŘESKOČENO, NAČTENO Z DRIVE: {modality}/{card['id']}")
                return saved_vector
            print(f"NEPLATNÝ / ZASTARALÝ CHECKPOINT, PŘEPOČÍTÁVÁM: {modality}/{card['id']}")
        except Exception as e:
            print(f"CHECKPOINT NELZE PŘEČÍST, PŘEPOČÍTÁVÁM {modality}/{card['id']}: {e}")

    print(f"POČÍTÁM: {modality}/{card['id']}")
    vector = model.encode(
        [item],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )[0]
    vector = np.asarray(vector, dtype=np.float32)
    if vector.ndim != 1 or not np.isfinite(vector).all():
        raise ValueError(f"Neplatný vektor: {modality}/{card['id']}")

    metadata = {
        "model": MODEL_ID,
        "modality": modality,
        "card_id": card["id"],
        "fingerprint": fingerprint,
        "dimension": int(vector.shape[0]),
        "created_at_unix": time.time(),
    }
    save_checkpoint(vector, vector_path, meta_path, metadata)
    print(f"ULOŽENO NA DRIVE: {modality}/{card['id']} · {vector.shape}")
    return vector

def build_vectors(modality, item_builder):
    result = []
    for card in records:
        result.append(encode_one(modality, card, item_builder(card)))
    return np.stack(result)

ids = [r["id"] for r in records]
text_vectors = build_vectors("text", lambda r: {"text": r["text"]})
image_vectors = build_vectors("image", lambda r: {"image": r["image"]})
joint_vectors = build_vectors("joint", lambda r: {"text": r["text"], "image": r["image"]})

assert text_vectors.shape == image_vectors.shape == joint_vectors.shape
assert all(np.isfinite(v).all() for v in [text_vectors, image_vectors, joint_vectors])

np.savez_compressed(
    OUTPUT_DIR / "embeddings.npz",
    ids=np.array(ids),
    text=text_vectors,
    image=image_vectors,
    joint=joint_vectors
)
print("\nVšech 9 embeddingů je připraveno.")
print("Text:", text_vectors.shape, "| Obraz:", image_vectors.shape, "| Společně:", joint_vectors.shape)

## ETAPA 6 · Analýza a vizualizace

Tato část pracuje s již uloženými vektory. Po novém připojení není potřeba znovu počítat embeddingy: spusť přípravné buňky, načti model jen pokud budeš chtít pokračovat ve výpočtu, a pro analýzu načti vektory z Drive.

In [ ]:
# 7. Načtení uložených vektorů pro analýzu
data = np.load(OUTPUT_DIR / "embeddings.npz")
ids = data["ids"].tolist()
text_vectors = data["text"]
image_vectors = data["image"]
joint_vectors = data["joint"]

assert len(ids) == 3
assert text_vectors.shape == image_vectors.shape == joint_vectors.shape
assert all(np.isfinite(x).all() for x in [text_vectors, image_vectors, joint_vectors])
print("Vektory načteny:", ids, text_vectors.shape)

## 8. Text proti ilustracím

Tady je jeden z hlavních pohledů. Každý řádek je text jedné pohádky, každý sloupec je ilustrace. Číslo je kosinová podobnost textového a obrazového embeddingu.

**Diagonála** (Ryba–Ryba, Myš–Myš atd.) ukazuje správně spárované dvojice. Ostatní políčka jsou kontrolní porovnání s ilustracemi jiných karet.

Samotná vysoká podobnost ještě neznamená, že obrázek „správně ilustruje“ celý příběh. Ukazuje jen blízkost v reprezentaci modelu.

In [ ]:
# 8. Křížová podobnost: text × obraz
text_image_sim = cosine_similarity(text_vectors, image_vectors)
sim_df = pd.DataFrame(text_image_sim, index=ids, columns=ids)

plt.figure(figsize=(max(12, len(ids)*0.38), max(9, len(ids)*0.32)))
sns.heatmap(sim_df, cmap="viridis", center=None)
plt.title("Qwen · podobnost českých textů a ilustrací")
plt.xlabel("Ilustrace")
plt.ylabel("Text pohádky")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

sim_df.to_csv(OUTPUT_DIR / "similarity_text_image.csv", encoding="utf-8-sig")

In [ ]:
# 9. Je vlastní ilustrace nejbližší?
# Pro každý text zjistíme pořadí jeho správné ilustrace mezi všemi ilustracemi.
rows = []
for i, card_id in enumerate(ids):
    scores = text_image_sim[i]
    order = np.argsort(scores)[::-1]
    rank = int(np.where(order == i)[0][0]) + 1
    rows.append({
        "karta": card_id,
        "podobnost_vlastni_ilustrace": float(scores[i]),
        "poradi_vlastni_ilustrace": rank,
        "nejpodobnejsi_ilustrace": ids[int(order[0])],
        "skore_nejpodobnejsi": float(scores[order[0]]),
        "rozdil_vlastni_minus_druha": float(scores[i] - scores[order[1]]) if len(order)>1 else np.nan,
    })

pairing_df = pd.DataFrame(rows).sort_values("poradi_vlastni_ilustrace")
display(pairing_df)
pairing_df.to_csv(OUTPUT_DIR / "text_image_pairing_ranks.csv", index=False, encoding="utf-8-sig")

print("Počet případů, kdy je vlastní ilustrace na 1. místě:",
      int((pairing_df["poradi_vlastni_ilustrace"] == 1).sum()), "/", len(pairing_df))

## 10. Podobnosti uvnitř každého typu reprezentace

Tyto tři matice odpovídají třem různým otázkám:
- Které pohádky jsou si podle modelu podobné textově?
- Které ilustrace jsou si podobné obrazově?
- Které karty jsou si podobné, když model dostane obojí společně?

Matice jsou v rámci stejného modelu a stejného prostoru. Nezaměňujeme je za lidské hodnocení významu.

In [ ]:
# 10. Tři matice podobností
text_sim = cosine_similarity(text_vectors)
image_sim = cosine_similarity(image_vectors)
joint_sim = cosine_similarity(joint_vectors)

def plot_matrix(matrix, title, filename):
    df = pd.DataFrame(matrix, index=ids, columns=ids)
    plt.figure(figsize=(max(10, len(ids)*0.34), max(8, len(ids)*0.30)))
    sns.heatmap(df, cmap="mako", vmin=-1, vmax=1)
    plt.title(title)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
    df.to_csv(OUTPUT_DIR / filename, encoding="utf-8-sig")
    return df

text_sim_df = plot_matrix(text_sim, "TEXT · podobnosti mezi pohádkami", "similarity_text_text.csv")
image_sim_df = plot_matrix(image_sim, "OBRAZ · podobnosti mezi ilustracemi", "similarity_image_image.csv")
joint_sim_df = plot_matrix(joint_sim, "SPOLEČNĚ · podobnosti mezi kartami", "similarity_joint_joint.csv")

## 11. Co se změní, když model dostane text a obraz společně?

Společný embedding porovnáme se samostatným textovým a obrazovým embeddingem stejné karty. Výsledky ukazují, ke kterému samostatnému vstupu je společná reprezentace v prostoru modelu blíž.

To není přímé měření „kolik nového významu vzniklo“. Je to měřitelná změna reprezentace při společném vstupu.

In [ ]:
# 11. Vztah společné reprezentace k samostatnému textu a obrazu
joint_to_text = np.sum(joint_vectors * text_vectors, axis=1)
joint_to_image = np.sum(joint_vectors * image_vectors, axis=1)
text_to_image = np.sum(text_vectors * image_vectors, axis=1)

fusion_df = pd.DataFrame({
    "karta": ids,
    "text_vs_obraz": text_to_image,
    "spolecne_vs_text": joint_to_text,
    "spolecne_vs_obraz": joint_to_image,
    "rozdil_spolecne_text_minus_obraz": joint_to_text - joint_to_image,
})
display(fusion_df.round(4))
fusion_df.to_csv(OUTPUT_DIR / "joint_representation_comparison.csv", index=False, encoding="utf-8-sig")

ax = fusion_df.set_index("karta")[["spolecne_vs_text", "spolecne_vs_obraz"]].plot(
    kind="bar", figsize=(max(12, len(ids)*0.35), 5)
)
ax.set_title("Ke které samostatné reprezentaci je společný embedding blíž?")
ax.set_ylabel("Kosinová podobnost")
ax.set_xlabel("Karta")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()

## 12. Vizuální mapa všech embeddingů

Pro přehled promítneme textové, obrazové a společné vektory do dvou rozměrů pomocí PCA. PCA zachovává jen část informace, proto je mapa orientační. Všechny tři typy vektorů promítáme společně, ne každou skupinu zvlášť.

Tím lze vizuálně sledovat, zda se reprezentace jedné karty drží pohromadě, nebo se text, obraz a společný vstup v mapě rozcházejí.

In [ ]:
# 12. PCA mapa vektorů
all_vectors = np.vstack([text_vectors, image_vectors, joint_vectors])
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(all_vectors)

n = len(ids)
plot_rows = []
for j, modality in enumerate(["text", "obraz", "společně"]):
    for i, card_id in enumerate(ids):
        plot_rows.append({
            "karta": card_id,
            "modalita": modality,
            "PC1": coords[j*n+i, 0],
            "PC2": coords[j*n+i, 1],
        })
pca_df = pd.DataFrame(plot_rows)

plt.figure(figsize=(13, 10))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="modalita", style="modalita", s=95)
for _, row in pca_df.iterrows():
    plt.annotate(row["karta"], (row["PC1"], row["PC2"]), fontsize=7, alpha=0.75)
plt.title("PCA · text, obraz a společné embeddingy")
plt.tight_layout()
plt.show()

pca_df.to_csv(OUTPUT_DIR / "pca_coordinates.csv", index=False, encoding="utf-8-sig")
print("Podíl vysvětlené variance:", pca.explained_variance_ratio_.round(4),
      "celkem:", round(float(pca.explained_variance_ratio_.sum()), 4))

## 13. Přehled rozsahu tří textů

Tento přehled uvádí délku pohádek po odstranění technických hlaviček a úvodního rámce před názvem. Není to skóre kvality ani významové hustoty.

In [ ]:
# 13. Přehled rozsahu textu (pouze kontext, nikoli skóre kvality)
length_df = pd.DataFrame([
    {
        "karta": r["id"],
        "znaky_vcetne_mezer": len(r["text"]),
        "slova_podle_mezer": len(r["text"].split()),
        "pocet_odstavcu": len([p for p in re.split(r"\n\s*\n", r["text"]) if p.strip()]),
    }
    for r in records
])
display(length_df.sort_values("znaky_vcetne_mezer"))
length_df.to_csv(OUTPUT_DIR / "text_length_context.csv", index=False, encoding="utf-8-sig")

## ETAPA 7 · Export ZIPu

Výsledný ZIP se ukládá na Google Drive. Obsahuje checkpointy, surové vektory, původní vstupy, metadata, matice podobností, pořadí párování, srovnání, PCA a kopie ilustrací.

In [ ]:
# 14. Manifest a export výsledků
images_out = OUTPUT_DIR / "images"
images_out.mkdir(exist_ok=True)

for card_id in ids:
    src = INPUT_DIR / f"{card_id}.jpg"
    if src.exists():
        (images_out / f"{card_id}.jpg").write_bytes(src.read_bytes())

metadata_rows = []
for card_id in ids:
    story = (INPUT_DIR / f"{card_id}.txt").read_text(encoding="utf-8")
    metadata_rows.append({
        "id": card_id,
        "text_file": f"{card_id}.txt",
        "image_file": f"{card_id}.jpg",
        "text_characters": len(story),
        "text": story
    })

(OUTPUT_DIR / "metadata.json").write_text(
    json.dumps(metadata_rows, ensure_ascii=False, indent=2), encoding="utf-8"
)

manifest = {
    "experiment": "MOUSE Multimodal Embeddings v1.2",
    "model": MODEL_ID,
    "language": "Czech",
    "cards": ids,
    "modalities": ["text_only", "image_only", "text_plus_image"],
    "vector_dimension": int(text_vectors.shape[1]),
    "device_used": device,
    "checkpoint_count": 9,
    "checkpoint_policy": "reuse only when model and input fingerprint match",
    "notes": [
        "One model is used for text, image and joint inputs.",
        "Each completed card/modality vector is checkpointed to Google Drive.",
        "Cosine similarity describes model representation geometry, not human-validated meaning.",
        "No MOUSE angle scores or semantic-density scores are fabricated."
    ]
}
(OUTPUT_DIR / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

zip_path = BASE / "MOUSE_Multimodal_Embeddings_v1_2_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in OUTPUT_DIR.rglob("*"):
        if file.is_file():
            z.write(file, file.relative_to(BASE))

print("HOTOVO:", zip_path)
print("Velikost MB:", round(zip_path.stat().st_size / 1024 / 1024, 2))
print("ZIP je uložený na Google Drive, i když se Colab odpojí.")

## Jak pokračovat po přerušení

1. Otevři znovu tento stejný notebook a připoj běhové prostředí.
2. Spusť buňku 1, potom buňku 2. Tím se znovu připojí Google Drive.
3. Spusť buňky 3 a 4, aby se obnovily vstupy a jejich kontrola.
4. Pokud nejsou hotové všechny embeddingy, spusť buňku 5 pro načtení modelu a potom buňku 6. Hotové checkpointy se vypíší jako **PŘESKOČENO, NAČTENO Z DRIVE** a znovu se nepočítají.
5. Pokud jsou všechny embeddingy hotové, můžeš přeskočit buňku 5 i výpočetní buňku 6 a pokračovat analýzou od buňky 7.
6. Nakonec spusť export ZIPu.

**Co se po přerušení opakuje:** připojení Drive a načtení modelu do RAM/VRAM. Pokud model ještě není v Drive cache, stáhne se. Pokud cache existuje, použije se. Samotný embedding, který byl dokončen a bezpečně uložen, se znovu nepočítá. Pokud se přerušení stane uprostřed právě počítaného vektoru, zopakuje se pouze tento jeden vektor.